# Lecture 16: NLP Applications & Case Studies — Answer Key
### NLP Course 2027 — Week 08

---
Complete answers for all four exercises in Lecture 16.

In [ ]:
import math, numpy as np
import matplotlib.pyplot as plt
from collections import Counter, defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords, reuters

for pkg in ['punkt', 'stopwords', 'reuters']:
    nltk.download(pkg, quiet=True)
print('Ready')

---
## Exercise 16.1 — BM25 vs TF-IDF Retrieval

**Task:** Implement BM25. Compare with TF-IDF on 5 queries over Reuters.

**BM25 formula:**
$$\text{score}(D,Q) = \sum_{t \in Q} \text{IDF}(t) \cdot \frac{f(t,D) \cdot (k_1+1)}{f(t,D) + k_1 \cdot (1 - b + b \cdot |D|/\text{avgdl})}$$

**Key concept:** BM25 adds length normalization over TF-IDF. Short documents shouldn't be unfairly penalized for having fewer total words. Parameter $k_1=1.5$ controls TF saturation; $b=0.75$ controls length normalization strength.

In [ ]:
class BM25:
    def __init__(self, corpus, k1=1.5, b=0.75):
        self.k1 = k1; self.b = b
        self.stop = set(stopwords.words('english'))
        self.tokenized = [self._tokenize(doc) for doc in corpus]
        self.N = len(self.tokenized)
        self.avgdl = np.mean([len(d) for d in self.tokenized])
        # Compute DF
        self.df = defaultdict(int)
        for doc in self.tokenized:
            for w in set(doc):
                self.df[w] += 1

    def _tokenize(self, text):
        return [w.lower() for w in word_tokenize(text)
                if w.isalpha() and w.lower() not in self.stop]

    def idf(self, term):
        df = self.df.get(term, 0)
        return math.log((self.N - df + 0.5) / (df + 0.5) + 1)

    def score(self, query, doc_tokens):
        dl = len(doc_tokens)
        tf = Counter(doc_tokens)
        score = 0.0
        for term in self._tokenize(query):
            f = tf.get(term, 0)
            score += self.idf(term) * (f * (self.k1 + 1)) / (
                f + self.k1 * (1 - self.b + self.b * dl / self.avgdl))
        return score

    def retrieve(self, query, top_k=3):
        scores = [self.score(query, doc) for doc in self.tokenized]
        return np.argsort(scores)[::-1][:top_k]


# Build corpus
doc_ids = reuters.fileids()[:100]
corpus  = [reuters.raw(fid) for fid in doc_ids]

bm25   = BM25(corpus)
tfidf  = TfidfVectorizer(stop_words='english').fit(corpus)
tfidf_matrix = tfidf.transform(corpus)

queries = [
    'oil price increase OPEC',
    'trade deficit import export',
    'interest rate Federal Reserve',
    'gold commodity price',
    'bank loan credit',
]

print(f'{"Query":<35} {"BM25 top-1 doc":<10} {"TF-IDF top-1 doc"}')
print('-' * 65)
for q in queries:
    bm25_top  = bm25.retrieve(q, 3)
    qvec      = tfidf.transform([q])
    tfidf_top = np.argsort(cosine_similarity(qvec, tfidf_matrix)[0])[::-1][:3]
    same = '✓ agree' if bm25_top[0] == tfidf_top[0] else '✗ differ'
    print(f'{q:<35} {doc_ids[bm25_top[0]]:<15} {doc_ids[tfidf_top[0]]:<15} {same}')

---
## Exercise 16.2 — LDA Topic Modeling on 20 Newsgroups

**Task:** Train LDA with K=20 topics. Do topics align with the 20 categories?

**Key concept:** LDA discovers topics as probability distributions over words. Topics often (but not always) correspond to human-defined categories. Sports and religion tend to be cleanest; politics often mixes with foreign policy.

In [ ]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

data = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
print(f'Loaded {len(data.data)} docs, {len(data.target_names)} categories')

vec = CountVectorizer(max_df=0.95, min_df=3, stop_words='english', max_features=5000)
dtm = vec.fit_transform(data.data)
vocab = vec.get_feature_names_out()

lda = LatentDirichletAllocation(n_components=20, random_state=42,
                                 max_iter=15, learning_method='online')
lda.fit(dtm)

print('\nTop 8 words per topic (compare with 20 Newsgroup category names):')
for topic_idx, topic in enumerate(lda.components_):
    top_words = [vocab[i] for i in topic.argsort()[-8:][::-1]]
    print(f'  Topic {topic_idx:2d}: {" | ".join(top_words)}')

print()
print('20 Newsgroup categories:', data.target_names)
print()
print('Observation: Topics about sports (hockey, baseball) and religion (god, church)')
print('are typically cleanest. Politics and tech often mix across topics.')

---
## Exercise 16.3 — TextRank vs BART Summarization

**Task:** Compare extractive TextRank vs abstractive BART on 5 articles.

**Key concept:** Extractive (TextRank) selects existing sentences — faithful but may be disjointed. Abstractive (BART) generates new text — more fluent but may hallucinate. Extractive wins on faithfulness; abstractive wins on conciseness and readability.

In [ ]:
def textrank_summarize(text, n_sentences=2):
    """TextRank extractive summarization."""
    sentences = sent_tokenize(text)
    if len(sentences) <= n_sentences:
        return text
    # TF-IDF similarity matrix between sentences
    vec = TfidfVectorizer(stop_words='english')
    try:
        sim_matrix = cosine_similarity(vec.fit_transform(sentences))
    except ValueError:
        return ' '.join(sentences[:n_sentences])
    np.fill_diagonal(sim_matrix, 0)
    # PageRank-style scoring
    scores = sim_matrix.sum(axis=1)
    top_idx = np.argsort(scores)[::-1][:n_sentences]
    top_idx_sorted = sorted(top_idx)  # preserve original order
    return ' '.join(sentences[i] for i in top_idx_sorted)


articles = [
    """Scientists at MIT have developed a new battery technology that could revolutionize
    electric vehicles. The new lithium-silicon battery offers three times the energy
    density of conventional lithium-ion batteries. The technology could allow EVs to
    travel up to 900 miles on a single charge. Researchers expect the technology to
    reach commercial production within five years. The breakthrough was published in
    the journal Nature Energy.""",
    """The global economy grew by 3.1% in the last quarter according to the IMF.
    Developing markets led the expansion with 5.2% growth. The United States posted
    2.4% GDP growth driven by consumer spending. Europe underperformed with only 1.1%
    growth amid energy concerns. China's recovery accelerated to 4.8% following
    stimulus measures.""",
]

try:
    from transformers import pipeline
    summarizer = pipeline('summarization', model='facebook/bart-large-cnn')
    use_bart = True
except Exception:
    use_bart = False

for i, article in enumerate(articles, 1):
    print(f'--- Article {i} ---')
    print(f'Original ({len(article.split())} words)')
    extractive = textrank_summarize(article, n_sentences=2)
    print(f'TextRank ({len(extractive.split())} words): {extractive}')
    if use_bart:
        abstractive = summarizer(article, max_length=50, min_length=20, do_sample=False)[0]['summary_text']
        print(f'BART     ({len(abstractive.split())} words): {abstractive}')
    print()

---
## Exercise 16.4 — Movie Recommendation with TF-IDF

**Task:** Build a content-based movie recommender using TF-IDF on plot descriptions.

**Key concept:** Content-based filtering represents items as feature vectors (TF-IDF of text). Cosine similarity between item vectors measures content overlap. This works well for cold-start (new movies with no ratings) but ignores user preferences.

In [ ]:
movies = [
    {'title': 'The Matrix',              'genre': 'sci-fi',
     'plot': 'A hacker discovers reality is a simulation and joins a rebellion against machines.'},
    {'title': 'Inception',               'genre': 'sci-fi',
     'plot': 'A thief who enters dreams to steal secrets is given the task of planting an idea.'},
    {'title': 'Interstellar',            'genre': 'sci-fi',
     'plot': 'Astronauts travel through a wormhole near Saturn to find a new home for humanity.'},
    {'title': 'The Dark Knight',         'genre': 'action',
     'plot': 'Batman must accept one of the greatest psychological and physical tests of his ability to fight injustice.'},
    {'title': 'Pulp Fiction',            'genre': 'crime',
     'plot': 'The lives of two mob hitmen a boxer and a pair of diner bandits interweave in tales of violence.'},
    {'title': 'The Shawshank Redemption','genre': 'drama',
     'plot': 'Two imprisoned men bond over a number of years finding solace and redemption through acts of decency.'},
    {'title': 'Avengers: Endgame',       'genre': 'action',
     'plot': 'The Avengers and their allies must be willing to sacrifice all in order to defeat Thanos.'},
    {'title': 'Arrival',                 'genre': 'sci-fi',
     'plot': 'A linguist works with the military to communicate with alien life forms that arrive on Earth.'},
]

titles = [m['title'] for m in movies]
plots  = [m['plot']  for m in movies]

vec    = TfidfVectorizer(stop_words='english')
tfidf  = vec.fit_transform(plots)
sim    = cosine_similarity(tfidf)

def recommend(title, top_k=3):
    idx    = titles.index(title)
    scores = sim[idx].copy()
    scores[idx] = -1  # exclude self
    top    = np.argsort(scores)[::-1][:top_k]
    return [(titles[i], f'{scores[i]:.3f}') for i in top]

for query in ['The Matrix', 'Pulp Fiction', 'Arrival']:
    recs = recommend(query)
    print(f'Because you watched "{query}":')
    for title, score in recs:
        print(f'  → {title}  (similarity={score})')
    print()

---
## Summary

| Exercise | Key Concept |
|----------|-------------|
| 16.1 | BM25 adds length normalization over TF-IDF; k1 controls TF saturation |
| 16.2 | LDA topics ≈ categories for clean subjects (sports/religion); politics mixes |
| 16.3 | Extractive = faithful copy; abstractive = fluent rephrasing — different trade-offs |
| 16.4 | TF-IDF content filtering: great for cold start; ignores user taste |

---
*NLP Course 2027 — Week 08*

---
**Author: Lei Wu | © 2026 Lei Wu. All rights reserved. Unauthorized use is prohibited.**